# 01 — Exploratory Data Analysis (EDA)

Khám phá bộ dữ liệu MovieLens trước khi xây dựng model.

**Nội dung:**
1. Load raw data
2. Rating distribution
3. User activity analysis
4. Item popularity analysis
5. Genre analysis
6. Temporal trends
7. Sparsity & cold-start analysis

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings

warnings.filterwarnings("ignore")

import logging

logging.disable(logging.CRITICAL)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from src.config import settings
from src.data.preprocessing import compute_sparsity, filter_cold_start, load_raw_data

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("Project root:", PROJECT_ROOT)

## 1. Load Raw Data

In [ ]:
ratings_raw, movies_raw, tags_raw = load_raw_data(settings.data_raw_dir)

print(f"Ratings : {len(ratings_raw):>10,} rows  |  columns: {list(ratings_raw.columns)}")
print(f"Movies  : {len(movies_raw):>10,} rows  |  columns: {list(movies_raw.columns)}")
print(f"Tags    : {len(tags_raw):>10,} rows  |  columns: {list(tags_raw.columns)}")
ratings_raw.head()

In [ ]:
print("Rating value counts:")
print(ratings_raw["rating"].value_counts().sort_index())
print(f"\nUnique users : {ratings_raw['userId'].nunique():,}")
print(f"Unique movies: {ratings_raw['movieId'].nunique():,}")

n_u = ratings_raw["userId"].nunique()
n_i = ratings_raw["movieId"].nunique()
raw_sparsity = compute_sparsity(len(ratings_raw), n_u, n_i)
print(f"Raw sparsity : {raw_sparsity:.4%}")

## 2. Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Rating Distribution", fontsize=13, fontweight="bold")

# Histogram
vc = ratings_raw["rating"].value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color="#4e9dd4", edgecolor="white")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")
axes[0].set_title("Rating Value Counts")
for x, y in zip(range(len(vc)), vc.values, strict=False):
    axes[0].text(
        x, y + max(vc.values) * 0.01, f"{y / 1e6:.1f}M", ha="center", va="bottom", fontsize=8
    )

# CDF
sorted_ratings = np.sort(ratings_raw["rating"].values)
cdf = np.arange(1, len(sorted_ratings) + 1) / len(sorted_ratings)
axes[1].plot(sorted_ratings, cdf, color="#e06c4b", lw=2)
axes[1].axvline(
    ratings_raw["rating"].mean(),
    color="grey",
    ls="--",
    label=f"Mean={ratings_raw['rating'].mean():.2f}",
)
axes[1].axvline(
    ratings_raw["rating"].median(),
    color="green",
    ls="--",
    label=f"Median={ratings_raw['rating'].median():.1f}",
)
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Cumulative Probability")
axes[1].set_title("Rating CDF")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_rating_distribution.png", bbox_inches="tight")
plt.show()

## 3. User Activity Analysis

In [ ]:
user_counts = ratings_raw.groupby("userId")["rating"].count().sort_values(ascending=False)

print(
    f"Ratings per user — mean: {user_counts.mean():.1f}, median: {user_counts.median():.0f}, max: {user_counts.max():,}"
)
print(f"Users with < 20 ratings : {(user_counts < 20).sum():,} ({(user_counts < 20).mean():.1%})")
print(
    f"Users with >= 100 ratings: {(user_counts >= 100).sum():,} ({(user_counts >= 100).mean():.1%})"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("User Activity", fontsize=13, fontweight="bold")

axes[0].hist(user_counts.values, bins=80, color="#4e9dd4", edgecolor="white", log=True)
axes[0].set_xlabel("Ratings per User")
axes[0].set_ylabel("Number of Users (log scale)")
axes[0].set_title("Distribution of User Activity")
axes[0].axvline(20, color="red", ls="--", label="min_user_ratings=20")
axes[0].legend()

# CCDF (Power-law check)
sorted_uc = np.sort(user_counts.values)[::-1]
axes[1].loglog(range(1, len(sorted_uc) + 1), sorted_uc, color="#e06c4b", lw=1.5)
axes[1].set_xlabel("User rank")
axes[1].set_ylabel("Number of ratings")
axes[1].set_title("User Activity — Log-Log (Power Law check)")

plt.tight_layout()
plt.savefig("eda_user_activity.png", bbox_inches="tight")
plt.show()

## 4. Item Popularity Analysis

In [ ]:
item_counts = ratings_raw.groupby("movieId")["rating"].count().sort_values(ascending=False)
item_avg = ratings_raw.groupby("movieId")["rating"].mean()

print(
    f"Ratings per movie — mean: {item_counts.mean():.1f}, median: {item_counts.median():.0f}, max: {item_counts.max():,}"
)
print(f"Movies with < 10 ratings : {(item_counts < 10).sum():,} ({(item_counts < 10).mean():.1%})")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Item Popularity", fontsize=13, fontweight="bold")

axes[0].hist(item_counts.values, bins=80, color="#74c476", edgecolor="white", log=True)
axes[0].set_xlabel("Ratings per Movie")
axes[0].set_ylabel("Number of Movies (log scale)")
axes[0].set_title("Distribution of Item Popularity")
axes[0].axvline(10, color="red", ls="--", label="min_item_ratings=10")
axes[0].legend()

# Top-20 most rated movies
top20 = item_counts.head(20).reset_index()
top20 = top20.merge(movies_raw[["movieId", "title"]], on="movieId", how="left")
top20["short_title"] = top20["title"].str[:30]
axes[1].barh(top20["short_title"][::-1], top20["rating"][::-1], color="#74c476")
axes[1].set_xlabel("Number of Ratings")
axes[1].set_title("Top 20 Most-Rated Movies")
axes[1].tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig("eda_item_popularity.png", bbox_inches="tight")
plt.show()

## 5. Genre Analysis

In [ ]:
# Explode pipe-delimited genres
genre_series = movies_raw["genres"].str.split("|").explode().str.strip()
genre_counts = genre_series[genre_series != "(no genres listed)"].value_counts()

print(f"Unique genres: {len(genre_counts)}")
print(genre_counts)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    genre_counts.index[::-1], genre_counts.values[::-1], color="#9e9ac8", edgecolor="white"
)
ax.set_xlabel("Number of Movies")
ax.set_title("Movie Count by Genre", fontsize=13, fontweight="bold")
for bar, val in zip(bars, genre_counts.values[::-1], strict=False):
    ax.text(
        bar.get_width() + 20,
        bar.get_y() + bar.get_height() / 2,
        f"{val:,}",
        va="center",
        fontsize=8,
    )
plt.tight_layout()
plt.savefig("eda_genre_counts.png", bbox_inches="tight")
plt.show()

In [ ]:
# Average rating by genre
genre_exploded = (
    movies_raw[["movieId", "genres"]]
    .assign(genre=movies_raw["genres"].str.split("|"))
    .explode("genre")
)
genre_exploded["genre"] = genre_exploded["genre"].str.strip()
genre_exploded = genre_exploded[genre_exploded["genre"] != "(no genres listed)"]

genre_ratings = (
    genre_exploded.merge(ratings_raw[["movieId", "rating"]], on="movieId", how="left")
    .groupby("genre")["rating"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = [
    "#4e9dd4" if v > genre_ratings["mean"].mean() else "#c0c0c0" for v in genre_ratings["mean"]
]
ax.barh(genre_ratings.index[::-1], genre_ratings["mean"][::-1], color=colors_bar[::-1])
ax.axvline(
    genre_ratings["mean"].mean(),
    color="red",
    ls="--",
    label=f"Mean={genre_ratings['mean'].mean():.2f}",
)
ax.set_xlabel("Average Rating")
ax.set_title("Average Rating by Genre", fontsize=13, fontweight="bold")
ax.legend()
ax.set_xlim(2.5, 4.5)
plt.tight_layout()
plt.savefig("eda_genre_avg_rating.png", bbox_inches="tight")
plt.show()

## 6. Temporal Trends

In [ ]:
ratings_raw["date"] = pd.to_datetime(ratings_raw["timestamp"], unit="s")
ratings_raw["year_month"] = ratings_raw["date"].dt.to_period("M")

monthly = ratings_raw.groupby("year_month").size()
monthly.index = monthly.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.suptitle("Temporal Trends", fontsize=13, fontweight="bold")

axes[0].plot(monthly.index, monthly.values, color="#4e9dd4", lw=1.2)
axes[0].fill_between(monthly.index, monthly.values, alpha=0.2, color="#4e9dd4")
axes[0].set_ylabel("Ratings per Month")
axes[0].set_title("Monthly Rating Volume")

# Average rating over time
monthly_avg = ratings_raw.set_index("date").resample("Q")["rating"].mean()
axes[1].plot(monthly_avg.index, monthly_avg.values, color="#e06c4b", lw=1.5, marker="o", ms=3)
axes[1].axhline(
    monthly_avg.mean(), color="grey", ls="--", label=f"Overall mean={monthly_avg.mean():.2f}"
)
axes[1].set_ylabel("Average Rating")
axes[1].set_title("Quarterly Average Rating Trend")
axes[1].legend()
axes[1].set_ylim(2.5, 5.0)

plt.tight_layout()
plt.savefig("eda_temporal_trends.png", bbox_inches="tight")
plt.show()

## 7. Sparsity & Cold-Start Analysis

In [ ]:
# Effect of cold-start filtering
thresholds = [(5, 5), (10, 5), (20, 10), (50, 20)]
rows = []
for min_u, min_i in thresholds:
    filtered = filter_cold_start(ratings_raw, min_u, min_i)
    nu = filtered["userId"].nunique()
    ni = filtered["movieId"].nunique()
    sp = compute_sparsity(len(filtered), nu, ni)
    rows.append(
        {
            "min_user_ratings": min_u,
            "min_item_ratings": min_i,
            "n_ratings": len(filtered),
            "n_users": nu,
            "n_items": ni,
            "sparsity": sp,
        }
    )

threshold_df = pd.DataFrame(rows)
print(threshold_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Effect of Cold-Start Filtering", fontsize=13, fontweight="bold")

labels = [f"({r['min_user_ratings']},{r['min_item_ratings']})" for _, r in threshold_df.iterrows()]

axes[0].bar(labels, threshold_df["n_ratings"] / 1e6, color="#4e9dd4", edgecolor="white")
axes[0].set_title("Ratings Remaining (M)")
axes[0].set_xlabel("(min_user, min_item)")

axes[1].bar(labels, threshold_df["n_users"] / 1e3, color="#fd8d3c", edgecolor="white")
axes[1].set_title("Users Remaining (K)")
axes[1].set_xlabel("(min_user, min_item)")

axes[2].bar(labels, threshold_df["sparsity"] * 100, color="#9e9ac8", edgecolor="white")
axes[2].set_title("Sparsity (%)")
axes[2].set_xlabel("(min_user, min_item)")
axes[2].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))

for ax in axes:
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("eda_cold_start_effect.png", bbox_inches="tight")
plt.show()
print(
    f"\nCurrent settings: min_user_ratings={settings.min_user_ratings}, min_item_ratings={settings.min_item_ratings}"
)